## **LangChain** - _Harry Potter - The Ultimate Trivia_

In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

C:\Users\danie\AppData\Local\Temp\ipykernel_24288\2492634325.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\danie\OneDrive\Documentos\GitHub\DAIXXRT\LangChain - Boost Academy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
# Dividimos en trozos de 900 caracteres con una superposición de 20
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,
                                               chunk_overlap = 150,
                                               separators = ["\n\n", "\n", " ", ""])

# Definimos el modelo de embeddings
modelo_embeddings = HuggingFaceEmbeddings(model = "BAAI/bge-m3")

# Conectamos e inicializamos nuestra colección en ChromaDB
vectorstore = Chroma(collection_name = "harry_potter_docs",
                     embedding_function = modelo_embeddings,
                     persist_directory = "../chroma_harry_potter")

# Ejecutar para insertar datos en la base de datos

# libros = [x for x in os.listdir(path = "../Data/harry_potter_books")]

# for libro in libros:

#     print(libro)
    
#     loader = TextLoader(file_path = f"../Data/harry_potter_books/{libro}", encoding = "utf-8")
#     documentos = loader.load()
#     fragmentos = text_splitter.split_documents(documents = documentos)
#     vectorstore.add_documents(documents = fragmentos)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8279.72it/s]


In [10]:
MODEL = "gemini-3.1-flash-lite"
TEMPERATURE = 0

llm = ChatGoogleGenerativeAI(model = MODEL, temperature = TEMPERATURE)
parser = StrOutputParser()

template = """
Usa los siguientes fragmentos de contexto recuperado para responder a la pregunta. 
Si no sabes la respuesta basándote en el contexto, di simplemente que no lo sabes.

Contexto:
{context}

Pregunta: {question}

Respuesta:
"""
prompt = PromptTemplate.from_template(template = template)

retriever = vectorstore.as_retriever(search_kwargs = {"k" : 10})

pregunta_usuario = "¿a que edad empieza harry estudiar en hogwarts?"

# Función auxiliar para unir los fragmentos en un solo texto
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Construir la cadena RAG (Pipeline)
cadena_rag = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

# Generar la respuesta final
respuesta_final = cadena_rag.invoke(input = pregunta_usuario)

print(respuesta_final)

Respuesta:

Basándome en el contexto proporcionado, Harry empieza a estudiar en Hogwarts a los once años. El fragmento menciona: "Cuatro veranos antes, el día en que cumplía once años, había entrado con Hagrid en la tienda del señor Ollivander para comprar una varita mágica" antes de iniciar su formación en el colegio.
